# 01 - Radiometric normalization

This notebook fits the transfer function that maps the render engine's photometric outputs to real physical irradiance. It splits the signal in two: an ambient term (environment lighting) and a geometric term fitted with polynomial regression (the solar-disc intensity). The result converts the simulated data into global horizontal irradiance (GHI).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Normalization configuration

In [ ]:
ARCHIVO = 'normalization_clearsky_il1.csv'

COL_DIRECTA_SIM   = 'sim_comp_direct_lux'      # Geometric solar component
COL_DIFUSA_SIM    = 'sim_comp_amb_lux'         # Integrated environmental component

COL_GHI_REAL      = 'clearsky_ghi_wm2'    # Real global irradiance value (final target)
COL_DHI_REAL      = 'clearsky_dhi_wm2'    # Real diffuse irradiance value

GRADO_POLINOMIO   = 2                 # For the geometric solar component (1=Linear, 2=Queadratic)
FORZAR_CERO       = True              # True = If there is darkness, the energy is 0

## 2. Load and filter

In [ ]:
print(f"--- LOADING {ARCHIVO} ---")
df = pd.read_csv(ARCHIVO)

mask = (df['altitude_deg'] > 5) & (df[COL_GHI_REAL] > 0)
df_clean = df[mask].copy()

print(f"Valid rows for calibration: {len(df_clean)} records")

## 3. Fitting the integrated ambient component

In [ ]:
X_diff = df_clean[[COL_DIFUSA_SIM]]
y_diff = df_clean[COL_DHI_REAL]

model_diff = LinearRegression(fit_intercept=not FORZAR_CERO)
model_diff.fit(X_diff, y_diff)

coef_difuso = model_diff.coef_[0]
print(f"\n[FASE A] Coeficiente Ambiental (Sombra): {coef_difuso:.6f}")

## 4. Fitting the geometric (solar) component

In [ ]:
y_residuo = df_clean[COL_GHI_REAL] - (coef_difuso * df_clean[COL_DIFUSA_SIM])

X_dir = df_clean[[COL_DIRECTA_SIM]]

poly = PolynomialFeatures(degree=GRADO_POLINOMIO, include_bias=False)
X_dir_poly = poly.fit_transform(X_dir) 

model_dir = LinearRegression(fit_intercept=not FORZAR_CERO)
model_dir.fit(X_dir_poly, y_residuo)

coefs_directos = model_dir.coef_
print(f"[FASE B] Coeficientes Directos (Sol): {coefs_directos}")

## 5. Normalization results

In [ ]:
y_pred_difusa = coef_difuso * df_clean[COL_DIFUSA_SIM]
y_pred_directa = model_dir.predict(X_dir_poly)
y_pred_total = (y_pred_directa + y_pred_difusa).clip(lower=0) 

r2 = r2_score(df_clean[COL_GHI_REAL], y_pred_total)
mae = mean_absolute_error(df_clean[COL_GHI_REAL], y_pred_total)
rmse = np.sqrt(mean_squared_error(df_clean[COL_GHI_REAL], y_pred_total))

print("\n" + "="*60)
print(f" FINAL RESULT — ADDITIVE MODEL")
print(f" R² Global: {r2:.5f}")
print(f" MAE: {mae:.2f} W/m² | RMSE: {rmse:.2f} W/m²")
print("="*60)

termino_difuso = f"{coef_difuso:.6f} * df['{COL_DIFUSA_SIM}']"
terminos_directos = []

if len(coefs_directos) >= 1:
    terminos_directos.append(f"{coefs_directos[0]:.6f} * df['{COL_DIRECTA_SIM}']")

if len(coefs_directos) >= 2:
    terminos_directos.append(f"{coefs_directos[1]:.8f} * df['{COL_DIRECTA_SIM}']**2")

formula_str = " + ".join(terminos_directos) + " + " + termino_difuso

print("\n>>> Python formula:")
print("-" * 80)
print(f"df['sim_ghi_wm2'] = (\n    {formula_str}\n).clip(lower=0)")
print("-" * 80)


## 6. Visualising the fit

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df_clean[COL_GHI_REAL], y_pred_total, alpha=0.5, color='blue', edgecolors='none', label='Data')
plt.plot([0, 1000], [0, 1000], 'r--', lw=2, label='Ideal (1:1)')
plt.xlabel("GHI Real (W/m²)")
plt.ylabel("GHI Simulado (W/m²)")
plt.title(f"Hybrid calibration: R²={r2:.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()